# Module 3 — Hybrid search: BM25 + k-NN

Vectors are great at meaning ("I woke up with a fever" finds the sick-leave policy) and weak at exact tokens ("E-1042" vs "E-1043" look almost identical to an embedding model). BM25 is the opposite. **Hybrid search** runs both and combines the scores.

In OpenSearch, the standard way is:

1. A **search pipeline** with a `normalization-processor`, which rescales BM25 and k-NN scores to a common range and combines them.
2. A **`hybrid` query** that contains the BM25 sub-query and the k-NN sub-query, sent with `?search_pipeline=<name>`.

We first check whether this works on our **OpenSearch Serverless NextGen** collection. If it does not, we say so and use the workaround: run the two queries separately and fuse the scores in Python with the same math.

Run Module 2 first: this notebook reuses its index.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from opensearchpy.exceptions import OpenSearchException
from tabulate import tabulate

from src.common import (
    HYBRID_WEIGHTS,
    INDEX_NAME,
    QUERY_KINDS,
    SEARCH_PIPELINE,
    TOP_K,
    get_opensearch_client,
    load_queries,
    min_max_fuse,
    recall_at_k,
)
from src.embeddings import get_embedder

CANDIDATES = 10  # hits each sub-query contributes before fusion

client = get_opensearch_client()
queries = load_queries()
embedder = get_embedder()
query_vectors = embedder.embed([q["text"] for q in queries])
print("embedding provider:", embedder.name, "| docs in index:", client.count(index=INDEX_NAME)["count"])

## Step 1 — BM25 alone and k-NN alone

A `match` query on the `text` field is BM25. We keep `(id, score)` pairs because we need the raw scores for fusion later.

In [ ]:
def bm25(text, size=TOP_K):
    body = {"size": size, "_source": False, "query": {"match": {"text": text}}}
    return [(h["_id"], h["_score"]) for h in client.search(index=INDEX_NAME, body=body)["hits"]["hits"]]


def knn(vector, size=TOP_K):
    body = {"size": size, "_source": False, "query": {"knn": {"embedding": {"vector": vector, "k": size}}}}
    return [(h["_id"], h["_score"]) for h in client.search(index=INDEX_NAME, body=body)["hits"]["hits"]]


print("BM25 for 'E-1042':", bm25("What does error E-1042 mean?"))
print("k-NN for 'E-1042':", knn(query_vectors[0]))

## Step 2 — Create the search pipeline

The `normalization-processor` runs between the query phase and the fetch phase:

- `normalization.technique: min_max` rescales each sub-query's scores to 0..1 (BM25 scores are unbounded, k-NN scores are not, so they cannot be added directly).
- `combination.technique: arithmetic_mean` with `weights` `[BM25, k-NN]` combines them. We start with equal weights.

The AWS docs list `PUT _search/pipeline/<id>` as a supported Serverless operation (it needs `aoss:CreateCollectionItems` in the data-access policy), so we expect this call to succeed.

In [ ]:
pipeline = {
    "description": "Min-max normalize BM25 and k-NN scores, then weighted mean",
    "phase_results_processors": [
        {
            "normalization-processor": {
                "normalization": {"technique": "min_max"},
                "combination": {"technique": "arithmetic_mean", "parameters": {"weights": list(HYBRID_WEIGHTS)}},
            }
        }
    ],
}

PIPELINE_CREATED = False
try:
    client.transport.perform_request("PUT", f"/_search/pipeline/{SEARCH_PIPELINE}", body=pipeline)
    PIPELINE_CREATED = True
    print("pipeline created:", client.transport.perform_request("GET", f"/_search/pipeline/{SEARCH_PIPELINE}"))
except OpenSearchException as exc:
    print("Could not create the search pipeline:", exc)

## Step 3 — Try the `hybrid` query

The `hybrid` query holds up to five sub-queries. Each returns its own candidates; the pipeline normalizes and merges them. If the collection does not support it, OpenSearch returns an error and we record that clearly.

In [ ]:
def hybrid_server(text, vector):
    body = {
        "size": TOP_K,
        "_source": False,
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"text": text}},
                    {"knn": {"embedding": {"vector": vector, "k": CANDIDATES}}},
                ]
            }
        },
    }
    response = client.search(index=INDEX_NAME, body=body, params={"search_pipeline": SEARCH_PIPELINE})
    return [(h["_id"], h["_score"]) for h in response["hits"]["hits"]]


SERVER_HYBRID_SUPPORTED = False
if PIPELINE_CREATED:
    try:
        print("hybrid for 'E-1042':", hybrid_server(queries[0]["text"], query_vectors[0]))
        SERVER_HYBRID_SUPPORTED = True
    except OpenSearchException as exc:
        print("hybrid query failed:", exc)

print()
if SERVER_HYBRID_SUPPORTED:
    print("RESULT: the hybrid query with a normalization-processor pipeline WORKS on this collection.")
else:
    print("RESULT: the hybrid query is NOT supported on this collection. Using the client-side workaround below.")

## Step 4 — The workaround: fuse scores on the client

If the server-side hybrid query is not available, we can get the same behavior in a few lines (`min_max_fuse` in `src/common.py`):

1. Run BM25 and k-NN as two normal searches, each returning `CANDIDATES` hits.
2. Min-max normalize each list to 0..1.
3. Weighted mean with the same weights. A doc missing from one list gets 0 for that list.

Costs: two round trips instead of one, and we normalize over the hits we fetched rather than over each shard's results. For a top-3 RAG retrieval on a small index, that difference is negligible. We run it even when the server-side version works, so we can compare.

In [ ]:
def hybrid_client(text, vector):
    return min_max_fuse([bm25(text, CANDIDATES), knn(vector, CANDIDATES)], weights=HYBRID_WEIGHTS)


print("client-side hybrid for 'E-1042':", hybrid_client(queries[0]["text"], query_vectors[0]))

## Step 5 — Compare the four approaches on every query

Recall@3 per query kind. Keyword queries (error codes, product names) should favor BM25; semantic paraphrases should favor k-NN; hybrid should be the most balanced.

In [ ]:
modes = {"BM25": lambda t, v: bm25(t), "k-NN": lambda t, v: knn(v), "hybrid (client)": hybrid_client}
if SERVER_HYBRID_SUPPORTED:
    modes["hybrid (server)"] = hybrid_server

per_query = []
recall = {m: {k: [] for k in QUERY_KINDS} for m in modes}
for q, vec in zip(queries, query_vectors):
    row = [q["id"], q["kind"], q["text"][:45]]
    for name, search in modes.items():
        ids = [doc_id for doc_id, _ in search(q["text"], vec)]
        r = recall_at_k(ids, q["relevant_ids"])
        recall[name][q["kind"]].append(r)
        row.append(f"{r:.2f}")
    per_query.append(row)

print(tabulate(per_query, headers=["id", "kind", "query", *modes], tablefmt="github"))
print()
summary = [[name, *(f"{sum(v[k]) / len(v[k]):.2f}" for k in QUERY_KINDS)] for name, v in recall.items()]
print(tabulate(summary, headers=["mode", *QUERY_KINDS], tablefmt="github"))

> **What you learned**
> - BM25 and vectors fail on different queries: exact codes and acronyms vs. paraphrases.
> - OpenSearch hybrid search = a `hybrid` query plus a search pipeline with a `normalization-processor`.
> - Scores must be normalized (e.g. min-max) before combining, because BM25 and k-NN use different scales.
> - When a server feature is missing, the same fusion can run client-side with two queries and a few lines of Python.
> - The weights are a tuning knob; we measure, not guess.

Next: run `python benchmark.py` from the repo root.